# Fine-tune RoBERTa

Table 3. Needs a GPU.

In [ ]:
import sys
sys.path.insert(0, "..")

from embodiment import config as cfg

import json
import shutil
from datetime import datetime

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, load_dataset
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, EarlyStoppingCallback,
                          Trainer, TrainingArguments)

RUN_ID = datetime.now().strftime("%Y%m%d-%H%M")
OUT_DIR = cfg.MODEL_DIR / f"roberta-nocontext-{RUN_ID}"
BASE_MODEL = "roberta-base"
PATIENCE = 4

print("CUDA:", torch.cuda.is_available(), "| run", RUN_ID)

## Data

In [ ]:
ds = load_dataset("csv", data_files={"train": str(cfg.TRAIN),
                                     "dev": str(cfg.DEV),
                                     "test": str(cfg.TEST)})
ds = ds.remove_columns([c for c in ds["train"].column_names
                        if c not in ("text", "label")])

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, truncation_side="left")


def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=512)


ds = ds.map(tokenize, batched=True)
collator = DataCollatorWithPadding(tokenizer=tokenizer)
{split: ds[split].num_rows for split in ds}

## Train

In [ ]:
def metrics_fn(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    out = {"f1_micro": f1_score(labels, preds, average="micro",
                                zero_division=0)}
    for i, v in enumerate(f1_score(labels, preds, average=None,
                                   zero_division=0)):
        out[f"f1_{cfg.ID2LABEL[i]}"] = v
    return out


TRAIN_ARGS = {
    "output_dir": str(OUT_DIR), "learning_rate": 1e-5,
    "per_device_train_batch_size": 8, "per_device_eval_batch_size": 8,
    "save_strategy": "epoch", "save_total_limit": 1,
    "load_best_model_at_end": True, "eval_strategy": "epoch",
    "num_train_epochs": 30,          # upper bound; early stopping ends the run
    "metric_for_best_model": "f1_micro",
    "log_level": "error", "report_to": "none",
}

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=len(cfg.LABELS), id2label=cfg.ID2LABEL,
    label2id=cfg.LABEL2ID)
trainer = Trainer(
    model=model, args=TrainingArguments(**TRAIN_ARGS),
    train_dataset=ds["train"], eval_dataset=ds["dev"], data_collator=collator,
    compute_metrics=metrics_fn,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)])
trainer.train()

## Test

In [ ]:
# Read the selected epoch before predict() appends its own eval entry.
dev_log = [e for e in trainer.state.log_history if "eval_f1_micro" in e]
best_dev = max(dev_log, key=lambda e: e["eval_f1_micro"])

pred = trainer.predict(ds["test"], metric_key_prefix="eval")
y_true, y_pred = pred.label_ids, np.argmax(pred.predictions, axis=1)

print(f"selected epoch {best_dev['epoch']:.0f} "
      f"(dev micro-F1 {best_dev['eval_f1_micro']:.4f})")
print(classification_report(y_true, y_pred,
                            target_names=[cfg.ID2LABEL[i] for i in
                                          sorted(cfg.ID2LABEL)],
                            zero_division=0))

## Bootstrap CIs

In [ ]:
N_BOOT = 1000
rng = np.random.default_rng(cfg.SEED)
ids = sorted(cfg.ID2LABEL)

boot = {"f1_micro": [], "f1_macro": [], **{cfg.ID2LABEL[i]: [] for i in ids}}
for _ in range(N_BOOT):
    idx = rng.integers(0, len(y_true), len(y_true))
    yt, yp = y_true[idx], y_pred[idx]
    boot["f1_micro"].append(f1_score(yt, yp, average="micro", zero_division=0))
    boot["f1_macro"].append(f1_score(yt, yp, average="macro", zero_division=0))
    for i, v in zip(ids, f1_score(yt, yp, labels=ids, average=None,
                                  zero_division=0)):
        boot[cfg.ID2LABEL[i]].append(v)

ci95 = {k: [float(lo), float(hi)] for k, (lo, hi) in
        ((k, np.percentile(v, [2.5, 97.5])) for k, v in boot.items())}
micro = f1_score(y_true, y_pred, average="micro", zero_division=0)
macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
print(f"micro {micro:.3f} [{ci95['f1_micro'][0]:.3f}, {ci95['f1_micro'][1]:.3f}]")
print(f"macro {macro:.3f} [{ci95['f1_macro'][0]:.3f}, {ci95['f1_macro'][1]:.3f}]")

## Save

In [ ]:
best_dir = cfg.MODEL_DIR / f"roberta-nocontext-{RUN_ID}-best"
trainer.save_model(str(best_dir))
tokenizer.save_pretrained(str(best_dir))

saved = dict(pred.metrics)
saved.update({"epoch": best_dev["epoch"],
              "dev_f1_micro": best_dev["eval_f1_micro"],
              "metrics_split": "test", "f1_macro": float(macro),
              "ci95": ci95})
(best_dir / "eval_metrics.json").write_text(json.dumps(saved, indent=2))
shutil.rmtree(OUT_DIR, ignore_errors=True)
print(f"saved -> {best_dir}")

## Cross-validation

In [ ]:
import gc

FOLDS = 5
gold = pd.read_csv(cfg.GOLD)
gold["label"] = gold["label"].map(cfg.LABEL2ID)


def quiet_metrics(eval_pred):
    logits, labels = eval_pred
    return {"f1_micro": f1_score(labels, np.argmax(logits, axis=1),
                                 average="micro", zero_division=0)}


def as_dataset(frame):
    return Dataset.from_pandas(frame[["text", "label"]],
                               preserve_index=False).map(tokenize, batched=True)


skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=cfg.SEED)
cv_true, cv_pred, fold_scores, cv_rows = [], [], [], []

for fold, (trainval_idx, test_idx) in enumerate(
        skf.split(gold, gold["label"]), start=1):
    trainval, fold_test = gold.iloc[trainval_idx], gold.iloc[test_idx]
    fold_train, fold_dev = train_test_split(
        trainval, test_size=0.15, random_state=cfg.SEED,
        stratify=trainval["label"])

    args = dict(TRAIN_ARGS)
    args["output_dir"] = str(cfg.MODEL_DIR / f"cv-{RUN_ID}-fold{fold}")
    fold_model = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL, num_labels=len(cfg.LABELS), id2label=cfg.ID2LABEL,
        label2id=cfg.LABEL2ID)
    fold_trainer = Trainer(
        model=fold_model, args=TrainingArguments(**args),
        train_dataset=as_dataset(fold_train), eval_dataset=as_dataset(fold_dev),
        data_collator=collator, compute_metrics=quiet_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=PATIENCE)])
    fold_trainer.train()

    out = fold_trainer.predict(as_dataset(fold_test))
    yt, yp = out.label_ids, np.argmax(out.predictions, axis=1)
    conf = torch.softmax(torch.tensor(out.predictions), dim=-1).numpy().max(axis=1)

    fold_scores.append({
        "fold": fold,
        "f1_micro": float(f1_score(yt, yp, average="micro", zero_division=0)),
        "f1_macro": float(f1_score(yt, yp, average="macro", zero_division=0))})
    cv_true.extend(yt.tolist())
    cv_pred.extend(yp.tolist())
    cv_rows.append(pd.DataFrame({
        "filename": fold_test["filename"].values,
        "sentence_id": fold_test["sentence_id"].values,
        "text": fold_test["text"].values,
        "gold": [cfg.ID2LABEL[i] for i in yt],
        "pred": [cfg.ID2LABEL[i] for i in yp],
        "confidence": conf.round(3), "fold": fold}))
    print(f"fold {fold}: micro {fold_scores[-1]['f1_micro']:.4f}")

    shutil.rmtree(args["output_dir"], ignore_errors=True)
    del fold_trainer, fold_model
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
fold_micro = [f["f1_micro"] for f in fold_scores]
fold_macro = [f["f1_macro"] for f in fold_scores]
print(f"CV micro-F1 {np.mean(fold_micro):.4f} +/- {np.std(fold_micro, ddof=1):.4f}")
print(f"CV macro-F1 {np.mean(fold_macro):.4f} +/- {np.std(fold_macro, ddof=1):.4f}")

# Every sentence is predicted exactly once, so pooled per-class F1 uses the
# full class counts rather than a single split's.
pooled = f1_score(cv_true, cv_pred, labels=ids, average=None, zero_division=0)

table3 = pd.DataFrame({
    "label": [cfg.ID2LABEL[i] for i in ids],
    "f1": [float(v) for v in f1_score(y_true, y_pred, labels=ids, average=None,
                                      zero_division=0)],
    "ci95": [ci95[cfg.ID2LABEL[i]] for i in ids],
    "cv_f1": [float(v) for v in pooled],
}).sort_values("f1", ascending=False)
table3.to_csv(cfg.TABLES / "table3_classifier.csv", index=False)

pd.concat(cv_rows, ignore_index=True).to_csv(cfg.CV_PREDICTIONS, index=False)
table3.round(3)